# People matching logistic model

Train a small supervised matcher from identifier-confirmed matches, identifier conflicts, comparison features, and missingness. Unlabelled pairs are excluded, and evaluation is conditional on pairs produced by the existing candidate blocking. The held-out split groups by the right-side person identifier.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_recall_curve
from sklearn.model_selection import GroupShuffleSplit

methodology_notebook = Path("notebooks/people_aggregation_simple.ipynb")
if not methodology_notebook.exists():
    methodology_notebook = Path("people_aggregation_simple.ipynb")
get_ipython().run_line_magic("run", str(methodology_notebook))

## Training data

In [ ]:
comparison_columns = [
    "same_normalised_name",
    "same_first_name",
    "first_names_compatible",
    "same_initials",
    "same_multiple_initials",
    "middle_initials_compatible",
    "same_birth_year",
    "same_death_year",
    "birth_year_conflict",
    "death_year_conflict",
    "same_country",
    "same_institution",
    "common_name",
    "unique_exact_name",
]
model_features = features[comparison_columns].astype(float).copy()

availability = {
    "first_name": people["first_name"].notna(),
    "initials": people["initials"].notna(),
    "birth_year": people["birth_year"].notna(),
    "death_year": people["death_year"].notna(),
    "country": people["countries"].map(bool),
    "institution": people["institution_names"].map(bool),
}
for name, available in availability.items():
    left_available = available.loc[left_keys].to_numpy()
    right_available = available.loc[right_keys].to_numpy()
    model_features[f"both_{name}_present"] = left_available & right_available
    model_features[f"one_{name}_missing"] = left_available ^ right_available

pair_keys = pd.Series(
    [frozenset(link) for link in features.index], index=features.index
)
labelled = model_features.copy()
labelled["pair_key"] = pair_keys
labelled = labelled[labelled["pair_key"].isin(known_pairs | hard_negative_pairs)]
labelled["match"] = labelled["pair_key"].isin(known_pairs).astype(int)
labelled["rule_match"] = features.loc[labelled.index, "predicted_match"].astype(int)

labelled["match"].value_counts().rename_axis("match").to_frame("pairs")

## Train and validate

In [ ]:
feature_columns = model_features.columns.tolist()
positions = np.arange(len(labelled))
groups = labelled.index.get_level_values(1)

split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_validation, test = next(
    split.split(positions, labelled["match"], groups)
)
validation_split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_relative, validation_relative = next(
    validation_split.split(
        train_validation,
        labelled.iloc[train_validation]["match"],
        groups[train_validation],
    )
)
train = train_validation[train_relative]
validation = train_validation[validation_relative]

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(
    labelled.iloc[train][feature_columns],
    labelled.iloc[train]["match"],
)

validation_probability = model.predict_proba(
    labelled.iloc[validation][feature_columns]
)[:, 1]
precision, recall, thresholds = precision_recall_curve(
    labelled.iloc[validation]["match"], validation_probability
)
f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1])
best_threshold = thresholds[np.nanargmax(f1)]
best_threshold

## Held-out evaluation

In [ ]:
def score(method, truth, prediction):
    true_negative, false_positive, false_negative, true_positive = confusion_matrix(
        truth, prediction, labels=[0, 1]
    ).ravel()
    precision = true_positive / (true_positive + false_positive)
    recall = true_positive / (true_positive + false_negative)
    return {
        "method": method,
        "true_positive": true_positive,
        "false_positive": false_positive,
        "false_negative": false_negative,
        "true_negative": true_negative,
        "precision": precision,
        "recall": recall,
        "f1": 2 * precision * recall / (precision + recall),
    }

test_rows = labelled.iloc[test]
test_probability = model.predict_proba(test_rows[feature_columns])[:, 1]
pd.DataFrame([
    score("current rules", test_rows["match"], test_rows["rule_match"]),
    score("logistic 0.5", test_rows["match"], test_probability >= 0.5),
    score(
        f"logistic validation F1 ({best_threshold:.3f})",
        test_rows["match"],
        test_probability >= best_threshold,
    ),
])

## Learned weights

In [ ]:
weights = pd.DataFrame({
    "feature": feature_columns,
    "weight": model.coef_[0],
})
weights["absolute_weight"] = weights["weight"].abs()
weights.sort_values("absolute_weight", ascending=False).head(20)

## Precision threshold sensitivity

In [ ]:
pd.DataFrame([
    score(
        f"threshold {threshold:.3f}",
        test_rows["match"],
        test_probability >= threshold,
    )
    for threshold in sorted({0.5, round(float(best_threshold), 3), 0.75, 0.9, 0.95})
])